# Article_forecast — Google Colab runner

Runs the demand-forecasting models on Colab. **Code** comes from GitHub; **large data** comes from your Google Drive.

## Before you start
1. **Runtime → Change runtime type → GPU** (needed for the LSTM).
2. Upload the data to Drive in a folder named `article_forecast_data` (or edit `DRIVE_DATA` below). It must contain:
   ```
   MyDrive/article_forecast_data/
   ├─ sequences/                 # all *_X_temporal.npy, *_X_static.npy, *_y.npy, eval_meta.npy
   ├─ feature_matrix.parquet
   ├─ merged.parquet
   ├─ encoders.json
   └─ scaler_params.json
   ```
3. Run the cells top to bottom.

In [ ]:
# --- 0. Check the GPU is on (Runtime > Change runtime type > GPU) ---
!nvidia-smi -L || echo 'No GPU — the LSTM will run on CPU (slow). Enable a GPU runtime.'

In [ ]:
# --- 1. Configuration (edit if your names differ) ---
REPO_URL   = 'https://github.com/alireza1420/Article_forecast.git'
BRANCH     = '006-lstm-demand-forecaster'
REPO       = '/content/Article_forecast'
DRIVE_DATA = '/content/drive/MyDrive/Alireza/article_forecast_data'        # where you uploaded the data
DRIVE_OUT  = '/content/drive/MyDrive/Alireza/article_forecast_results'     # where results get saved back

In [ ]:
# --- 2. Clone the code from GitHub ---
import os
if not os.path.exists(REPO):
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
%cd {REPO}
!git log --oneline -1

In [ ]:
# --- 3. Mount Google Drive (the large data lives here) ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- 4. Link the Drive data into the repo's data/processed/ ---
# (data/processed/ is gitignored, so the clone doesn't include it — we symlink it in.)
import os
os.makedirs(f'{REPO}/data/processed', exist_ok=True)
items = ['sequences', 'feature_matrix.parquet', 'merged.parquet',
         'encoders.json', 'scaler_params.json']
missing = []
for name in items:
    s = f'{DRIVE_DATA}/{name}'
    d = f'{REPO}/data/processed/{name}'
    if os.path.exists(s):
        os.system(f'ln -sfn "{s}" "{d}"')   # -f force, -n don't follow existing dir symlink
        print('linked  ', name)
    else:
        missing.append(s)
        print('MISSING ', s)
if missing:
    print('\n⚠️  Upload the missing files to Drive before running the models.')

In [ ]:
# --- 5. Create the output folders the scripts write to ---
for sub in ['results/figures', 'results/tables', 'results/models',
            'results/predictions']:
    os.makedirs(f'{REPO}/{sub}', exist_ok=True)
print('output dirs ready')

In [ ]:
# --- 6. Install dependencies (torch is already on Colab with CUDA — not in requirements.txt) ---
!pip install -q -r {REPO}/requirements.txt

In [ ]:
# --- 7. Sanity check: data is reachable and torch sees the GPU ---
%cd {REPO}
import sys, glob; sys.path.insert(0, 'src')
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
from features import load_feature_matrix
fm = load_feature_matrix()
print('feature_matrix:', fm.shape)
print('sequence .npy files found:', len(glob.glob('data/processed/sequences/*.npy')))

## Train + evaluate the ML models (XGBoost, RandomForest, LightGBM, CatBoost)

⚠️ **RandomForest is slow and memory-hungry** (it produced ~19 GB of checkpoints locally). If you only need the boosted trees + LSTM comparison, edit the `models = [...]` list in `src/ml_models.py`'s `__main__` to drop `RandomForestForecaster()` before running this cell.

In [ ]:
%cd {REPO}
!python src/ml_models.py

## Train + evaluate the LSTM

Order matters: `lstm_model.py` runs the ablation and writes `results/tables/lstm_arch_experiments.csv` + `results/models/lstm/lstm_final.pt`, which `evaluate.py` then reads. Both use paths **relative to the working directory**, so the `%cd {REPO}` is required.

In [ ]:
%cd {REPO}
!python src/lstm_model.py       # ablation -> lstm_final.pt + experiment log
!python src/evaluate.py         # evaluate best config, write figures + metric tables

In [ ]:
# --- Optional: IMS (iterated multi-step) LSTM variant ---
%cd {REPO}
!python src/lstm_ims_model.py

## Save results back to Drive

Colab storage is wiped when the session ends — copy what you want to keep. This excludes the giant `random_forest` checkpoint dump; everything else (figures, tables, predictions, metric CSVs, LSTM/XGB/LGBM checkpoints) is small enough to keep.

In [ ]:
import os
os.makedirs(DRIVE_OUT, exist_ok=True)
# rsync the results tree, skipping the 19 GB random_forest checkpoints
!rsync -a --exclude 'models/random_forest' "{REPO}/results/" "{DRIVE_OUT}/"
print('Results saved to', DRIVE_OUT)
!ls -R "{DRIVE_OUT}/figures" "{DRIVE_OUT}/tables" 2>/dev/null | head -40